# 003 函数文件与最小测试

沿用001校准资料，把每一步变为可以单独检查的函数。先读讲义与实验指南。

In [1]:
from pathlib import Path
import experiment as lab
import faults
from test_experiment import run_tests
base = Path.cwd()
training = lab.read_csv_records(base / "data/train.csv", ["id", "x", "y"])
print(training[0])
print(type(training[0]["x"]).__name__)

{'id': 'T01', 'x': 1.0, 'y': 3.0}
float


## 字典与嵌套查找
先选择第二条记录，再取x。保持原始记录不变。

In [2]:
rows = [{"id": "A", "x": 2}, {"id": "B", "x": 4}]
print(rows[1]["x"])
assert rows[1]["x"] == 4

4


## 实例各自保存偏移量
实例first和second共享类的写法，不共享各自的bias。

In [3]:
first = lab.AffineRule(1)
second = lab.AffineRule(2)
print(first.predict_one(3), second.predict_one(3))
inputs = [1.5, 2.5, 3.5, 4.5]
before = inputs.copy()
print(first.predict(inputs))
assert inputs == before

7 8
[4.0, 6.0, 8.0, 10.0]


## 先预测再评分
只用训练记录确定规则。

In [4]:
selection = lab.choose_bias(training)
print(selection)
lookup = lab.LookupRule(training)
print("命中", lookup.predict_one(3), "未命中", lookup.predict_one(2.5))
assert selection["bias"] == 1
assert lab.mean_absolute_error([3, 5], [2, 6]) == 1

{'bias': 1, 'scores': [{'bias': 0, 'train_mae': 1.2}, {'bias': 1, 'train_mae': 0.2}, {'bias': 2, 'train_mae': 0.8}]}
命中 8.0 未命中 0


## 故意制造索引与类型错误
只捕捉明确预期的异常类型。

In [5]:
try:
    faults.wrong_indices([1, 2, 3])
except IndexError:
    print("IndexError：已识别越界位置")
try:
    faults.unconverted_prediction("1.5")
except TypeError:
    print("TypeError：已识别未转换文本")
wrong = faults.wrong_denominator([3, 5], [2, 6])
print("错误除数结果", wrong, "手算应为", 1)
assert abs(wrong - 1) > 0.1

IndexError：已识别越界位置
TypeError：已识别未转换文本
错误除数结果 0.6666666666666666 手算应为 1


## 局部名字与共享对象
局部数字参数不会修改外部同名变量，修改共享列表对象则会被外部看到。

In [6]:
bias = 10
def add_offset(x, bias):
    result = 2 * x + bias
    return result
print(add_offset(3, 1), bias)
a = [1, 2]
b = a
b.append(3)
print(a)
assert bias == 10
assert a == [1, 2, 3]

7 10
[1, 2, 3]


## 全套测试与完整结果
测试在临时目录制造坏CSV和路径错误，另用新进程检查导入和启动目录。

In [7]:
test_result = run_tests()
print(test_result)
metrics = lab.run(base / "notebook_outputs")
print(metrics)
assert test_result["status"] == "passed"
assert metrics["A_test_mae"] == 0
assert metrics["B_test_mae"] == 7

{'status': 'passed', 'test_groups': ['three hand-calculated metric fixtures', 'empty, mismatched, nonfinite and text metric inputs rejected', 'instance isolation, inherited method and nonmutation', 'base interface and ID alignment', 'six malformed CSV cases rejected', 'three exception faults and a silent wrong denominator reproduced', 'integration metrics match independent hand fixtures', 'fresh-process import needs no data and creates no result files', 'same results from an unrelated working directory']}
{'selected_bias': 1, 'A_train_mae': 0.2, 'B_train_mae': 0.0, 'A_test_mae': 0.0, 'B_test_mae': 7.0, 'A_stress_mae': 3.0}


## 最后重启并运行全部
保留故障报告：最小输入、预期、实际、定位、修正、回归测试。测试通过支持这里检查过的行为，不证明任意输入都正确。